In [14]:
import  pandas as pd

In [15]:
df = pd.read_csv('IMDB Dataset.csv')

In [16]:
df.shape

(50000, 2)

In [17]:
df = df.drop_duplicates()

In [18]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [19]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [20]:
import re
def clean_text(text):
    text = text.lower()

    #remove url
    text = re.sub(r"http\s+", "", text)

    # panchuation
    text = re.sub(r"[^A-Za-z0-9\s]","", text)

    #html tag
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    #extra space
    text = re.sub(r"\s+", " ", text).strip()

    return text

df['review'] = df['review'].apply(clean_text)

In [21]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


## remove stop words

In [22]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/hemantpatidar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/hemantpatidar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hemantpatidar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [23]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [28]:
def remove_stopwords(text):

    tokens = word_tokenize(text)
    stop_words = stopwords.words('english')

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, '')

    return text

df['review'] = df['review'].apply(remove_stopwords)

In [29]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


## steamming

In [32]:
from nltk.stem import PorterStemmer

def stemming(text):

    ps = PorterStemmer()

    tokens = word_tokenize(text)

    stemmed_words = []

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)


    return " ".join(stemmed_words)

df['review'] = df['review'].apply(stemming)

In [33]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


# encoding

In [34]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

In [35]:
y = df['sentiment']

In [36]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


# vectorization 

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['review'])

In [39]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057200 stored elements and shape (49582, 5000)>

In [40]:
y.value_counts()

sentiment
1    24884
0    24698
Name: count, dtype: int64

In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [48]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [49]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [51]:
train_set = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train.values).float())
test_set = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test.values).float())

In [54]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

# build RNN

In [55]:
import torch.nn as nn
import torch.optim as optim

In [62]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)#(seq_len, batch_size, input_size) ==> batch, seq, input

        # fully connected

        self.fc = nn.Linear(hidden_size, 1)


    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        #1s hidden state of all the timestamp batch_size, sq, hidden_size

        out = self.fc(out[:,-1,:])#batch_size, sq, hidden_size

        return out

In [63]:
X_train.shape

(39665, 5000)

In [64]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [65]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1) # increase dimention
        output = model(xb)
        output = torch.sigmoid(output.squeeze())

        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

    print(f'{epoch+1/epochs} loss {loss.item()}')
        

0.1 loss 0.2712209224700928
1.1 loss 0.26873478293418884
2.1 loss 0.267799973487854
3.1 loss 0.39151430130004883
4.1 loss 0.1528901755809784
5.1 loss 0.21799658238887787
6.1 loss 0.28526026010513306
7.1 loss 0.20789113640785217
8.1 loss 0.1886824369430542
9.1 loss 0.3528375029563904


In [66]:
model.eval()

with torch.no_grad():

    correct =0
    total =0
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        output = model(xb)

        predict = (torch.sigmoid(output.squeeze()) > 0.5).float()

        total += yb.size(0)
        correct += (predict == yb).sum().item()


    print('accuracy ', correct/total)

accuracy  0.8572148835333266
